# Tarea 1

## Cuerpo docente

- Profesores: [Andrés Abeliuk](https://aabeliuk.github.io/), Fabrizzio Pezzolla
- Profesor auxiliar: Rodrigo Molina

## Instrucciones generales

- Grupos de máximo 4 personas.
- Está prohibido compartir las respuestas con otros grupos.
- Indicios de copia serán penalizados con la nota mínima.
- Cualquier duda fuera del horario de clases, al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente; si utilizan material extra, deben citarlo.


## Integrantes

> POR FAVOR AGREGAR TODOS LOS NOMBRES DE LOS INTEGRANTES

# 0. Preparación


In [ ]:
%%capture

!pip install wordcloud
!pip install gensim

import nltk
nltk.download('punkt_tab')


In [ ]:
import re

import numpy as np
import pandas as pd
from scipy.sparse import spmatrix

from nltk import FreqDist
from nltk import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

import matplotlib.pyplot as plt
from wordcloud import WordCloud

# word2vec
from gensim.models import Word2Vec, KeyedVectors


En esta sección cargaremos el dataset. Para ello, ejecuten la siguiente celda y reemplacen `SEMILLA` por el número de su grupo.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/dccuchile/CC6205/master/assignments/new/assignment_1/train/train.tsv", sep="\t")

SEMILLA = 0  # TODO: reemplacen por el número de su grupo


# 1. Detección de odio e incivilidad

El corpus para esta tarea se compone de 3 datasets:
- [Multilingual Resources for Offensive Language Detection de Arango et al. (2022)](https://aclanthology.org/2022.woah-1.pdf#page=136)
- [Dataton UTFSM No To Hate (2022)](http://dataton.inf.utfsm.cl/)
- Datos generados usando la [API de GPT-3 (modelo DaVinci 03)](https://platform.openai.com/docs/models/gpt-3).

> Agradecimientos a los autores por compartir los datos y a David Miranda, Fabián Diaz, Santiago Maass y Jorge Ortiz por revisar y reetiquetar los datos en el contexto del curso «Taller de Desarrollo de Proyectos de IA» (CC6409) del Departamento de Ciencias de la Computación, Universidad de Chile. Los datos solo pueden ser usados con fines de investigación y docencia. Está prohibida la difusión externa.

Consideren los siguientes conceptos:
- Se considera *discurso de odio* a cualquier expresión que promueva o incite a la discriminación, la hostilidad o la violencia hacia una persona o grupo de personas en una relación asimétrica de poder, tales como la raza, la etnia, el género, la orientación sexual, la religión, la nacionalidad, una discapacidad u otra característica similar.
- Se considera *incivilidad* a cualquier comportamiento o actitud que rompe las normas de respeto, cortesía y consideración en la interacción entre personas. Esta puede manifestarse de diversas formas, tales como insultos, ataques personales, sarcasmo, desprecio, entre otras.

En esta tarea tendrán a su disposición un dataset de textos con las etiquetas `odio`, `incivilidad` o `normal`. La mayor parte de los datos se encuentra en español de Chile. Se les pedirá explorar, analizar y visualizar el corpus para luego preparar representaciones vectoriales y entrenar un clasificador de manera supervisada.

## 1.1 Análisis exploratorio

Antes de trabajar con el texto, describan el corpus:

- Cuántos textos hay de cada clase.
- El largo de los textos de cada clase, en palabras separadas por espacios: promedio y mediana (pueden usar `describe()`). **(0,1 pts)**

In [ ]:
# TODO 1: número de textos por clase


In [ ]:
# TODO 2: largo de los textos por clase


---

**📝 Pregunta 1 (0,1 pts):** Describan cada clase en función de cuántas palabras contienen sus textos. ¿Sería suficiente información para entrenar un clasificador?
> **Respuesta:**

---

## 1.2 Etiquetas del corpus

Lean los siguientes cinco textos del corpus.

In [ ]:
# Cinco textos del corpus, sin su etiqueta
ids_ejercicio = [475, 3504, 10590, 1449, 3653]
ejercicio = df.set_index("id").loc[ids_ejercicio, "texto"]

for i, texto in ejercicio.items():
    print(f"[{i}] {texto}\n")

---

**📝 Pregunta 2 (0,4 pts):** Usando las definiciones anteriores de *discurso de odio* (`odio`) e *incivilidad* (`incivilidad`), asígnenle una etiqueta a cada texto antes de ver la del dataset. Consideren `normal` como textos que no corresponden ni a `odio` ni a `incivilidad`. Respondan con una etiqueta por texto, con su justificación.
> **Respuesta:**

---

A continuación, podrán observar todas las filas del dataset en que aparecen los textos anteriores.

In [ ]:
# Todas las filas con esos textos
df[df["texto"].isin(ejercicio)].sort_values("texto")[["id", "clase", "texto"]]


---

**📝 Pregunta 3 (0,5 pts):** Comparen sus predicciones con las etiquetas presentes en el dataset. ¿Qué podría haber llevado a que esas sean las etiquetas?
> **Respuesta:**

---

## 1.3 Tokenizador

En el siguiente ejemplo se muestra cómo usar el tokenizador `word_tokenize` de la librería `nltk`:

In [ ]:
# Elegimos un documento cualquiera del corpus
documento = df["texto"].sample(1, random_state=SEMILLA).values[0]

# Evaluamos el tokenizador word_tokenize
documento_procesado = word_tokenize(documento)

print(documento_procesado)

Construyan un tokenizador que reciba un string y retorne los tokens correspondientes al string. Su tokenizador debe contener al menos tres patrones. **(0,3 pts)**

In [ ]:
def tokenizador(_documento: str) -> list[str]:
    
    # TODO 3: su tokenizador
    
    return [] # Reemplazar

Pueden utilizar la siguiente celda para evaluar su tokenizador.

In [ ]:
# Elegimos un documento cualquiera del corpus
documento = df["texto"].sample(1, random_state=SEMILLA).values[0]
print(documento)

# Evaluamos nuestro tokenizador
documento_procesado = tokenizador(documento)

assert isinstance(documento_procesado, list)

print(documento_procesado)

---

**📝 Pregunta 4 (0,3 pts):** ¿Cómo funciona su tokenizador? Justifiquen su elección de expresiones regulares.
> **Respuesta:**

---

## 1.4 Exploración del vocabulario


Usando su tokenizador, generen un gráfico de frecuencias con todos los tokens del corpus. **(0,1 pts)**

In [ ]:
# TODO 4: gráfico de frecuencias con todos los tokens del corpus

Revisen su tokenizador a partir de lo que ven en el gráfico y defínanlo de nuevo en la celda siguiente. De aquí en adelante, usen `tokenizador_v2`. **(0,1 pts)**

In [ ]:
def tokenizador_v2(_documento: str) -> list[str]:

    # TODO 5: su tokenizador revisado

    return [] # Reemplazar

---

**📝 Pregunta 5 (0,3 pts):** Expliquen las diferencias entre su primer tokenizador y su segunda versión. ¿Qué decisiones tomaron a partir del gráfico?
> **Respuesta:**

---

Usando el gráfico de frecuencias, identifiquen **por lo menos** diez stopwords en el vocabulario. **(0,1 pts)**

In [ ]:
# TODO 6: sus stopwords
stopwords = ... # iterable

Ahora, usando su tokenizador y sus stopwords, generen una nube de palabras **para cada clase**. **(0,1 pts)**

In [ ]:
# TODO 7: una nube de palabras por clase (sin stopwords)


---

**📝 Pregunta 6 (0,1 pts):** ¿Cuáles palabras parecen ser representativas de cada clase? Mencionen tres palabras por clase.
> **Respuesta:**

---

## 1.5 Semántica vectorial


Desde aquí trabajaremos con una separación fija en entrenamiento y prueba, la misma para todos los grupos. Las representaciones se construyen solo con los textos de entrenamiento; los de prueba quedan reservados para evaluar los clasificadores en la sección 1.6.

In [ ]:
# Sin textos con etiquetas en conflicto ni duplicados
etiquetas_por_texto = df.groupby("texto")["clase"].nunique()
en_conflicto = etiquetas_por_texto[etiquetas_por_texto > 1].index
df_limpio = df[~df["texto"].isin(en_conflicto)].drop_duplicates("texto")

# La misma separación para todos los grupos
X_train, X_test, y_train, y_test = train_test_split(
    df_limpio["texto"], df_limpio["clase"],
    test_size=0.2, stratify=df_limpio["clase"], random_state=0,
)

Generen una representación TF-IDF de los textos de entrenamiento y de prueba, utilizando su `tokenizador_v2` y su lista de stopwords. Ajusten el vectorizador solo con `X_train` y usen `transform` para `X_test`. **(0,1 pts)**

In [ ]:
# TODO 8: TF-IDF
vectorizador = ...
tfidf_train = ...
tfidf_test = ...

Generen una representación LSA de 100 dimensiones a partir de sus matrices TF-IDF. **(0,1 pts)**

In [ ]:
# TODO 9: LSA
svd = TruncatedSVD(n_components=100, random_state=0)
lsa_train = ...
lsa_test = ...

La siguiente celda elige un tweet de entrenamiento con su semilla. Muestren sus 5 vecinos más cercanos entre los textos de entrenamiento (similitud coseno), con TF-IDF y con LSA. **(0,1 pts)**

In [ ]:
# Un tweet de entrenamiento elegido con su semilla
largos = X_train.str.replace("@user", "").str.split().str.len()
consulta = X_train[largos >= 8].sample(1, random_state=SEMILLA).index[0]
fila = X_train.index.get_loc(consulta)  # su fila en tfidf_train y lsa_train
print("Tweet de ejemplo:", X_train[consulta])

# TODO 10: 5 vecinos más cercanos con TF-IDF y con LSA


---

**📝 Pregunta 7 (0,5 pts):** ¿Cómo se comparan los vecinos de su tweet obtenidos al usar TF-IDF y LSA? ¿A qué se deben estas diferencias?
> **Respuesta:**

---

Entrenen Word2Vec con los textos de entrenamiento y representen cada documento como el promedio de los vectores de sus palabras. **(0,2 pts)**

In [ ]:
def vector_documento(tokens: list[str], kv: KeyedVectors) -> np.ndarray:
    """Promedia los vectores de las palabras de un documento.

    Ignora las palabras que no están en `kv`. Si no queda ninguna,
    devuelve un vector de ceros.
    """
    vectores = [kv[p] for p in tokens if p in kv]
    return np.mean(vectores, axis=0) if vectores else np.zeros(kv.vector_size)

# workers=1 para que el resultado sea reproducible
parametros = dict(sg=1, vector_size=100, window=5, min_count=5, negative=10, epochs=5, seed=0, workers=1)

# TODO 11: Word2Vec y vectores de documento
w2v = ...
emb_train = ...
emb_test = ... 


---

**📝 Pregunta 8 (0,4 pts):**

**(a) (0,2 pts)** Muestren los 10 vecinos más cercanos de `mapuche`, `inmigrantes`, `fachos` y `comunistas` en su Word2Vec. ¿Son palabras similares a la original o palabras relacionadas con ella? ¿Qué dicen estos vecinos sobre el corpus?

**(b) (0,2 pts)** `fachos` y `comunistas` designan bandos políticos opuestos: ¿quedan cerca en su Word2Vec? ¿Por qué?


In [ ]:
# TODO 12: vecinos de mapuche, inmigrantes, fachos y comunistas


> **Respuesta (a):**

> **Respuesta (b):**

---

## 1.6 Clasificadores


### 1.6.1 Entrenamiento

Entrenen un clasificador de regresión logística con cada representación, usando `tfidf_train`, `lsa_train` y `emb_train` con `y_train`. **(0,1 pts)**

In [ ]:
tfidf_clf = LogisticRegression(max_iter=2000)
lsa_clf = LogisticRegression(max_iter=2000)
emb_clf = LogisticRegression(max_iter=2000)

# TODO 13: entrenar los tres clasificadores


### 1.6.2 Evaluación

La siguiente celda calcula la exactitud (*accuracy*), es decir, la fracción de textos bien clasificados, de una línea base que predice siempre la clase más frecuente del conjunto de entrenamiento.

In [ ]:
# Línea base: la clase más frecuente
clase_mayoritaria = y_train.mode()[0]
print(clase_mayoritaria, (y_test == clase_mayoritaria).mean())


Completen la función `evaluar`, que construye la matriz de confusión (filas: clase real; columnas: predicción). A partir de ella, calculen el *recall* de cada clase, la fracción de los textos de esa clase que el modelo clasificó correctamente (la diagonal dividida por la suma de su fila), y la exactitud (la suma de la diagonal dividida por el total). **(0,2 pts)**

In [ ]:
def evaluar(
    clf: LogisticRegression, X: spmatrix | np.ndarray, y: pd.Series
) -> pd.Series:
    """Muestra la matriz de confusión y calcula recall y exactitud.

    Devuelve una Series con el recall de cada clase, seguido de la
    exactitud.
    """
    matriz = pd.crosstab(
        y, clf.predict(X), rownames=["real"], colnames=["predicción"]
    )
    # Las mismas clases en filas y columnas
    columnas = matriz.index.rename("predicción")
    matriz = matriz.reindex(columns=columnas, fill_value=0)
    display(matriz)

    # TODO 14: recall de cada clase y exactitud, a partir de `matriz`
    recall = ...  # una Series
    exactitud = ...

    return pd.concat([recall, pd.Series({"exactitud": exactitud})])

In [ ]:
resultados = {}
for nombre, clf, X in [("TF-IDF", tfidf_clf, tfidf_test), ("LSA", lsa_clf, lsa_test), ("Embeddings", emb_clf, emb_test)]:
    print(nombre)
    resultados[nombre] = evaluar(clf, X, y_test)

pd.DataFrame(resultados).round(3)

Identifiquen la clase con menor *recall* en su mejor clasificador (el de mayor exactitud). Muestren cinco textos de prueba de esa clase que el modelo asignó a la clase con la que más se confunde. **(0,1 pts)**

In [ ]:
# TODO 15: cinco errores del mejor clasificador


---

**📝 Pregunta 9 (0,5 pts):** ¿Cuál es la clase con menor *recall* en su mejor clasificador y con cuál la confunde más? Lean los cinco errores que mostraron: para cada uno, ¿se equivocó el modelo o la etiqueta es discutible, como en la sección 1.2?
> **Respuesta:**

---

# 2. Preguntas finales


---

**📝 Pregunta 10 (0,5 pts):** Imaginen que una red social quiere usar su modelo para moderar sus interacciones. ¿Qué error consideran peor: `odio` → `normal` o `normal` → `odio`? ¿Qué cambiarían?
> **Respuesta:**

---

**📝 Pregunta 11 (0,7 pts):** Las palabras `negro` y `perra` tienen más de un sentido en el corpus.

**(a) (0,4 pts)** Para cada palabra, muestren tres textos con su clase (con al menos dos sentidos distintos). Muestren también sus 10 vecinos más cercanos en su Word2Vec. ¿A qué sentidos corresponden esos vecinos?

**(b) (0,3 pts)** ¿Puede la representación vectorial (TF-IDF, LSA, Word2Vec) de un tweet distinguir el sentido con el que se está usando `negro` o `perra`? ¿Por qué?


In [ ]:
# TODO 16: textos y vecinos de negro y perra


> **Respuesta (a):**

> **Respuesta (b):**

---